# Financial Transaction ETL Pipeline
Coding by Justine Berger - October 2026

The purpose of this project is to demonstrate a full Extract, Transform, and Load process.

### Project Workflow
1. Extract raw CSV datasets
2. Perform exploratory data profiling
3. Identify data quality issues
4. Clean and standardize data using Python
5. Export cleaned datasets
6. Load data into PostgreSQL
7. Validate relationships and business rules using SQL
8. Prepare data for business intelligence reporting

In [2]:
# LOAD LIBRARIES
import pandas as pd
import numpy as np

# Extract 
## Import Data
The data being used for this project was obtained from Kaggle: https://www.kaggle.com/datasets/testdatabox/finance-fraud-and-loans-dataset-testdatabox/data

It is a series of csv files with some of the tables containing messy data. 
 
Data dictionary included at: https://github.com/DataJustine/Financial-Transaction-ETL/blob/main/docs/Data_Dictionary.pdf

In [ ]:
# Load datasets
account_statuses = pd.read_csv('account_statuses.csv')
account_types = pd.read_csv('account_types.csv')
accounts = pd.read_csv('accounts.csv')
addresses = pd.read_csv('addresses.csv')
branches = pd.read_csv('branches.csv')
customer_types = pd.read_csv('customer_types.csv')
customers = pd.read_csv('customers.csv')
loan_statuses = pd.read_csv('loan_statuses.csv')
loans = pd.read_csv('loans.csv')
transaction_types = pd.read_csv('transaction_types.csv')
transactions = pd.read_csv('transactions.csv')

# Initial Data Exploration

In this section, we will explore the data in each table to see if there are any issues we need to address in the cleaning process.

### Table 1 - account_statuses

In [4]:
# Look at the size of the data
account_statuses.shape

(3, 2)

Note that the size of the account_statuses table is very small (3 rows, 2 columns), so looking at all the data in a glance for this table is feasible.

In [5]:
# Look at the data
account_statuses

,AccountStatusID,StatusName
0,1,Active
1,2,Inactive
2,3,Closed


Looking at the data it is very basic. There does not appear to be any nulls, duplicates, consistency issues etc. Thus, this data appears clean and ready to go to SQL.

### Table 2 - account_types

In [6]:
# Look at the size of the data
account_types.shape

(5, 2)

Again the size is very small (5 x 2), so let's view the data for a quick scan.

In [7]:
# Look at the data
account_types

,AccountTypeID,TypeName
0,1,Checking
1,2,Savings
2,3,Payroll
3,4,Business
4,5,Youth


Looking at the data it is very basic. There does not appear to be any nulls, duplicates, consistency issues etc. Thus, this data appears clean and ready to go to SQL. 

### Table 3 - accounts

In [8]:
# Look at the size of the data
accounts.shape

(1667, 6)

This data is a bit larger (1,667 x 6) so let's peek at a few records.

In [9]:
# Look at data
accounts.head(10)

,AccountID,CustomerID,AccountTypeID,AccountStatusID,Balance,OpeningDate
0,200094,10123,3,1,48348.54,2018-06-12 00:00:00.000000
1,201108,10077,3,1,35001.41,2019-10-30 00:00:00.000000
2,201453,10321,3,2,57081.03,2020-05-24 00:00:00.000000
3,200581,10871,5,1,63164.33,2021-01-27 00:00:00.000000
4,200003,10765,1,1,58739.64,2018-09-12 00:00:00.000000
5,201097,10903,1,1,45430.05,2021-07-18 00:00:00.000000
6,201343,10681,5,1,15435.59,2018-07-24 00:00:00.000000
7,201250,11038,5,1,52166.08,2018-07-10 00:00:00.000000
8,201001,10655,4,2,14569.86,2021-07-09 00:00:00.000000
9,200342,10672,4,1,76080.75,2019-06-04 00:00:00.000000


The OpeningDate is in a timestamp format, however, the time is always 0's. Consider formating as just date.

The timestamp for our analysis purposes most likely will not be important, therefore, this appears to be an appropriate step to take. This will be done in the cleaning (transform) section.

Could there be nulls and duplicates within the table or could the OpeningDate have invalid dates? Let's look.

In [10]:
# Review for nulls and review types of data.
accounts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1667 entries, 0 to 1666
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   AccountID        1667 non-null   int64  
 1   CustomerID       1667 non-null   int64  
 2   AccountTypeID    1667 non-null   int64  
 3   AccountStatusID  1667 non-null   int64  
 4   Balance          1667 non-null   float64
 5   OpeningDate      1634 non-null   object 
dtypes: float64(1), int64(4), object(1)
memory usage: 78.3+ KB


In [11]:
accounts.isnull().sum()

AccountID           0
CustomerID          0
AccountTypeID       0
AccountStatusID     0
Balance             0
OpeningDate        33
dtype: int64

Noted previously this table had 1,667 rows, thus, looking at the info here there are 33 nulls located in the OpeningDate (1667-1634). 

There could be various reasons for this like, but not limited to, they are in the process of opening and not yet fully opened or could be a possible manual keying error. 

The null could be an appropriate value, therefore, we will leave it as is for now.

POSSIBLE RECOMMENDATION: Have the system not allow a blank open field or to automatically add the date when the record was created. Depends on exactly what this populates from, how important this date is and where it is being used.


In [12]:
# Review for duplicates
accounts.duplicated().value_counts()

False    1651
True       16
Name: count, dtype: int64

It does appear there are 16 duplicated lines.

In [13]:
# Review duplicates to see if they appear to be legitimate
accounts[accounts.duplicated(keep=False)].sort_values(by=['AccountID'])

,AccountID,CustomerID,AccountTypeID,AccountStatusID,Balance,OpeningDate
843,200002,10839,5,1,84008.77,2019-11-15 00:00:00.000000
362,200002,10839,5,1,84008.77,2019-11-15 00:00:00.000000
549,200118,10298,4,1,61554.44,2021-09-10 00:00:00.000000
556,200118,10298,4,1,61554.44,2021-09-10 00:00:00.000000
869,200147,10931,3,1,741.98,2018-10-16 00:00:00.000000
1354,200147,10931,3,1,741.98,2018-10-16 00:00:00.000000
252,200186,10686,2,1,75229.57,2021-09-13 00:00:00.000000
447,200186,10686,2,1,75229.57,2021-09-13 00:00:00.000000
479,200332,10908,1,2,7437.98,2021-02-12 00:00:00.000000
692,200332,10908,1,2,7437.98,2021-02-12 00:00:00.000000


These do appear to be legitimate duplicates. The duplicated lines will be removed during the cleaning process.

In [13]:
# Look for invalid dates
pd.to_datetime(accounts['OpeningDate'], errors='coerce').dt.date.isna().value_counts()

OpeningDate
False    1634
True       33
Name: count, dtype: int64

33 invalid dates align with the 33 null values previously noted. Thus, it does not appear to be any additional invalid dates other than the original nulls.

In [14]:
# Look at values in each field to make sure it makes sense
accounts.describe()

,AccountID,CustomerID,AccountTypeID,AccountStatusID,Balance
count,1667.000000,1667.000000,1667.000000,1667.000000,1667.000000
mean,200823.184763,10550.444511,3.046791,1.244151,49072.473935
std,476.914013,311.305260,1.410037,0.526393,29191.917303
min,200000.000000,10000.000000,1.000000,1.000000,-486.680000
25%,200411.500000,10292.500000,2.000000,1.000000,23838.620000
50%,200821.000000,10543.000000,3.000000,1.000000,49112.500000
75%,201235.500000,10818.500000,4.000000,1.000000,74783.450000
max,201650.000000,11099.000000,5.000000,3.000000,99828.980000


Figures appear appropriate. Nothing is giving indication of issues. 

Noted a negative in the Balance column information, however, it is possible someone has overdrawn their account. Therefore, we will leave as is.

In [15]:
# Check squence of AccountID

## Create expected range
accountID_range = set(range(accounts['AccountID'].min(),accounts['AccountID'].max() + 1))

## Compare against actual
missing_accountIDs = accountID_range - set(accounts['AccountID'])

## View any missing
print(f"Missing AccountIDs: {list(missing_accountIDs)}")

Missing AccountIDs: []


AccountID sequence appears to have no missing values. No action needed. 

### Table 4 - addresses

In [16]:
# Review the size of the data
addresses.shape

(1222, 4)

This is a good sized dataset (1,222 x 4).

In [17]:
# Review for nulls and review types of data.
addresses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1222 entries, 0 to 1221
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   AddressID  1222 non-null   int64 
 1   Street     1198 non-null   object
 2   City       1196 non-null   object
 3   Country    1198 non-null   object
dtypes: int64(1), object(3)
memory usage: 38.3+ KB


Nulls noted. 

Data types appear appropriate.

In [17]:
# Count of nulls
addresses.isnull().sum()

AddressID     0
Street       24
City         26
Country      24
dtype: int64

The addresses table represents customers and branches. Null values could be addresses in the process of being set up.

Some could be related to the accounts that are null. Leaving null values as is unless further analysis later supports an update.

In [18]:
# View lines with any null present
addresses[addresses.isnull().any(axis=1)]

,AddressID,Street,City,Country
3,780,Avon,Universal City,NaN
6,414,High,NaN,United States
24,411,NaN,Munster,United States
62,326,NaN,Garden City,United States
109,1011,Carson,NaN,United States
...,...,...,...,...
1167,372,Bitting,Weston,NaN
1170,896,El Camino Del Mar,Salinas,NaN
1186,146,Crisp,Ridgecrest,NaN
1190,582,NaN,Laurel,United States


Nothing stands out at the moment.

In [20]:
# View lines with all nulls present
addresses[addresses.isnull().all(axis=1)]

,AddressID,Street,City,Country


No lines with all nulls noted. This gives indication that only some of the information is missing. This supports leaving nulls and does not support dropping any observations as the partial data could be useful and important to maintain.

In [19]:
# View unique values assess inconsistencies and assess possible Null value updates needed
addresses['Country'].value_counts()

Country
United States    1186
Unitd States        3
United StateR       1
United Slates       1
United vtates       1
United Staes        1
UnitedcStates       1
United State        1
United StXtes       1
United0States       1
Pnited States       1
Name: count, dtype: int64

Noted inconsistent data i.e. misspellings. There is a need to clean United States to be the same correct value. Possible that the Nulls need to be United States as well. 

For the purpose of this project, we will assume that this company does not have any branches or customers out of the country. Typically, we would confirm with the Business to ensure this is the case in a production environment.

In [20]:
# View unique values assess inconsistencies and assess possible Null value updates needed
addresses['City'].value_counts()

City
Laconia               5
Country Club Hills    5
Leawood               4
Mount Pleasant        4
St. Matthews          4
                     ..
Xenia                 1
Danville              1
Whittier              1
Lynnwood              1
Milwaukie             1
Name: count, Length: 870, dtype: int64

Nothing is jumping out as an issue at the moment.

In [24]:
# Review for duplicates
addresses.duplicated().value_counts()

False    1210
True       12
Name: count, dtype: int64

It does appear there are 12 duplicated lines. These will be cleaned up in the cleaning process i.e. 6 removed.

In [ ]:
# Sequence Check
## Create expected range
AddressID_range = set(range(addresses['AddressID'].min(),addresses['AddressID'].max() + 1))

## Compare actual to expected 
missing_AddressIDs = AddressID_range - set(addresses['AddressID'])

## View missing sequences
print(f"Missing AddressIDs: {list(missing_AddressIDs)}")

No missing AddressIds.

### Table 5 - branches

In [21]:
# Review the size of the data
branches.shape

(50, 3)

This is a small dataset (50 x 3). We could scan through and look at the data manually, however, we will continue with Python to analyze issues, if any.

In [22]:
# Review for nulls and review types of data.
branches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   BranchID    50 non-null     int64 
 1   BranchName  50 non-null     object
 2   AddressID   50 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 1.3+ KB


Noted no nulls and data types appear appropriate.

In [23]:
# Double check for null values
branches.isnull().sum()

BranchID      0
BranchName    0
AddressID     0
dtype: int64

No nulls noted

In [24]:
# Review for duplicates
branches.duplicated().value_counts()

False    50
Name: count, dtype: int64

It does not appear there are duplicated lines in the branches table.

In [25]:
# Sequence Check

## Create expected range
BranchID_range = set(range(branches['BranchID'].min(),branches['BranchID'].max() + 1))

## Compare actual to expected 
missing_BranchIDs = BranchID_range - set(branches['BranchID'])

## View missing sequences
print(f"Missing BranchIDs: {list(missing_BranchIDs)}")


Missing BranchIDs: []


Does not appear to be any missing BranchID sequences.

### Table 6 - customer_types

In [26]:
# Look at the size of the data
customer_types.shape

(3, 2)

Small amount of data (3 x 2). Let's manually review.

In [ ]:
# View data
customer_types

No duplicates.
No Nulls.
No inconsistencies.
Nothing raising concern.

### Table 7 - customers

In [27]:
# View size of data
customers.shape

(1111, 6)

This is a good sized dataset (1,111 x 6). Continue with further analysis via Python.

In [28]:
# Review for nulls and data types
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1111 entries, 0 to 1110
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   CustomerID      1111 non-null   int64 
 1   FirstName       1089 non-null   object
 2   LastName        1088 non-null   object
 3   DateOfBirth     1111 non-null   object
 4   AddressID       1111 non-null   int64 
 5   CustomerTypeID  1111 non-null   int64 
dtypes: int64(3), object(3)
memory usage: 52.2+ KB


Nulls are present.
All formatting appears appropriate.

In [129]:
# Count of nulls
customers.isnull().sum()

CustomerID         0
FirstName         22
LastName          23
DateOfBirth        0
AddressID          0
CustomerTypeID     0
dtype: int64

NULLS present in FirstName and LastName columns only. This indicates that there is other data in the records that could be important. It does not appear appropriate to remove records with nulls.

Since data is related to bank transactions, it does not appear appropriate for first nor last names to be omitted. Potential recommendation noted.

In [30]:
# View the lines with null values
customers[customers.isnull().any(axis=1)]

,CustomerID,FirstName,LastName,DateOfBirth,AddressID,CustomerTypeID
1,10983,NaN,Battle,1963-02-01 00:00:00.000000,958,2
12,10233,Nisha,NaN,1999-09-15 00:00:00.000000,591,1
23,10830,NaN,Burt,1992-06-29 00:00:00.000000,1169,2
55,10161,NaN,Pena,1975-03-23 00:00:00.000000,634,1
92,10145,NaN,Rogers,1985-03-22 00:00:00.000000,869,2
129,10179,NaN,James,1972-05-09 00:00:00.000000,235,3
151,10525,Lucio,NaN,1967-11-13 00:00:00.000000,1060,1
154,10699,Claudio,NaN,1983-10-15 00:00:00.000000,614,2
178,11006,Deshawn,NaN,1973-01-20 00:00:00.000000,55,2
181,10355,Jacquiline,NaN,1960-09-18 00:00:00.000000,1107,3


Noted invalid DateOfBirth 'NaT'. Will need to clean and replace with Null.

Noted other invalid date formats. Will need to clean for consistency.

Nothing else jumping out at the moment. 

Nulls will be left as is.

In [ ]:
# Look for invalid dates
pd.to_datetime(customers['DateOfBirth'], errors='coerce').dt.date.isna().value_counts()

DateOfBirth
False    1079
True       32
Name: count, dtype: int64

Noting there are 32 invalid dates this includes 'NaT'.

In [32]:
# View the invalid dates
customers[pd.to_datetime(customers['DateOfBirth'], errors='coerce').dt.date.isna()]

,CustomerID,FirstName,LastName,DateOfBirth,AddressID,CustomerTypeID
75,10384,Mammie,Faulkner,1960-08-27T00:00:00,988,2
124,10626,Johnson,Lawrence,NaT,143,2
199,10226,Thurman,NaN,NaT,1020,2
205,10713,Gilbert,Bernard,NaT,1139,1
278,10653,Faviola,Robertson,NaT,1067,2
279,10505,Audrie,Juarez,NaT,779,1
296,10139,Lynna,NaN,07.04.1981,439,1
302,10365,Ariel,Parks,1985/07/09,1001,2
321,10500,Chauncey,Ward,02/11/1970,1062,2
421,10515,Eloy,Chaney,NaT,373,3


Dates will need to be cleaned for consistency within the table as well as with other tables.

In [33]:
# Review for value issues
customers.describe()

,CustomerID,AddressID,CustomerTypeID
count,1111.000000,1111.000000,1111.000000
mean,10550.559856,615.470747,2.044104
std,318.454163,347.599343,0.823549
min,10000.000000,1.000000,1.000000
25%,10274.500000,323.000000,1.000000
50%,10551.000000,622.000000,2.000000
75%,10826.500000,921.500000,3.000000
max,11099.000000,1210.000000,3.000000


The min and max on CustomerID does not appear to be appropriate when compared to count. Most likely missing sequences and/or duplicate values.

When looking at min and max versus count for AddressID, it is possible that AddressID is associated with more than one customer as they could be residing in the same residence or using the same branch.

No other items noted that raises concern.

In [ ]:
# Sequence Check
## Create expected range
CustomerID_range = set(range(customers['CustomerID'].min(),customers['CustomerID'].max() + 1))

## Compare actual to expected 
missing_CustomerIDs = CustomerID_range - set(customers['CustomerID'])

## View missing sequences
print(f"Missing CustomerIDs: {list(missing_CustomerIDs)}")


No missing Ids.

Missing sequences can be indication of deleted transactions. This could be inadvertent or purposeful. Regardless if missing values present in unique ID fields that should be in sequence, investigation would need to be conducted to ensure reasons are appropriate.

In [34]:
# Check for duplicates
customers.duplicated().sum()

np.int64(11)

11 duplicates noted. These will be cleaned.

In [36]:
# View duplicates
##customers[customers.duplicated(keep=False)].sort_values(by=['CustomerID'])

### Table 8 - loan_statuses

In [ ]:
# Look at the size of the data
loan_statuses.shape

(3, 2)

dataset size is small (3, 2) let's review manually.

In [ ]:
# View the data
loan_statuses

No duplicates.
No Nulls.
No inconsistencies.
Nothing raising concern.

### Table 9 - loans

In [38]:
# Look at the shape
loans.shape

(333, 7)

Decent size (333 x 7) we will review using Python.

In [ ]:
# Look at the data of some observations
loans.head(10)

StartDate and EstimatedEndDate to be reformatted in cleaning process to be consistent with other dates in the datasets. 

Need to assess the same fields for invalid dates.

In [39]:
# View the invalid dates
loans[pd.to_datetime(loans['StartDate'], errors='coerce').dt.date.isna()]

,LoanID,AccountID,LoanStatusID,PrincipalAmount,InterestRate,StartDate,EstimatedEndDate
129,400314,200982,1,74680.25,0.0870,NaN,2024-08-31 00:00:00.000000
177,400183,200619,1,68151.05,0.1453,NaN,2024-06-18 00:00:00.000000
195,400270,200824,1,77144.41,0.0456,NaN,2024-09-27 00:00:00.000000
212,400052,200433,1,84790.28,0.1027,NaN,2025-01-27 00:00:00.000000
224,400229,200120,1,21375.07,0.1136,NaN,2024-09-26 00:00:00.000000
325,400037,200498,1,84938.36,0.1452,NaN,2025-03-06 00:00:00.000000


Only NaN noted. No further cleaning needed after formatting. 

If able, I would want to look at documentation to get the StartDate or inquire with the business. 

Currently, there is no information that would help determine what the field should be nor for imputing a value. Leaving as is.

In [40]:
# View the invalid dates
loans[pd.to_datetime(loans['EstimatedEndDate'], errors='coerce').dt.date.isna()]

,LoanID,AccountID,LoanStatusID,PrincipalAmount,InterestRate,StartDate,EstimatedEndDate
117,400232,200683,2,30825.35,0.1469,2023-02-09 00:00:00.000000,NaN
128,400318,200941,2,43704.85,0.0446,2022-03-18 00:00:00.000000,NaN
205,400043,201360,1,64131.66,0.0950,2023-02-26 00:00:00.000000,NaN
229,400325,200397,1,24534.48,0.1021,2026-08-29 15:01:44.338382,NaN
245,400105,201562,1,81783.10,0.0480,2022-08-15 00:00:00.000000,NaN
288,400155,200078,2,27406.57,0.0415,2022-07-24 00:00:00.000000,NaN


Only NaN noted. No further cleaning needed after formatting.

In [ ]:
# Review for validity the StartDate vs EstimatedEndDate as Start should be prior to End not after
loans[loans['StartDate'] > loans['EstimatedEndDate']]

,LoanID,AccountID,LoanStatusID,PrincipalAmount,InterestRate,StartDate,EstimatedEndDate
262,400080,201058,1,41196.54,0.1429,2026-03-18 15:01:44.338382,2024-05-30 00:00:00.000000


Noted one record with StartDate prior to EndDate. We will look at imputing the EstimatedEndDate from average days between when doing clean up.

If able, I would want to look at documentation for this loan or inquire with the business to be able to update with the correct value.

The start and end date of the loan is important for calculating interest revenue for the bank.

In [ ]:
# Review data types and null values
loans.info()

There are nulls and dates will be reformatted.

In [42]:
# Count of nulls
loans.isnull().sum()

LoanID              0
AccountID           0
LoanStatusID        0
PrincipalAmount     0
InterestRate        0
StartDate           6
EstimatedEndDate    6
dtype: int64

Nulls are only in the Date fields, which will not be updated as there is other information in the observations that could be important and useful.

In [ ]:
# Sequence Check
## Create expected range
LoanID_range = set(range(loans['LoanID'].min(),loans['LoanID'].max() + 1))

## Compare actual to expected 
missing_LoanIDs = LoanID_range - set(loans['LoanID'])

## View missing sequences
print(f"Missing LoanIDs: {list(missing_LoanIDs)}")


No missing Ids.

In [ ]:
# Check for duplicates
loans.duplicated().value_counts()

Duplicates noted and need to be removed during cleaning.

In [ ]:
# Check stats of data
loans.describe()

### Table 10 - transaction_types

In [52]:
# View data size
transaction_types.shape

(4, 2)

Dataset is small (4 x 2). Review manually using Python.

In [53]:
# View data
transaction_types

,TransactionTypeID,TypeName
0,1,Deposit
1,2,Withdrawal
2,3,Transfer
3,4,Payment


No duplicates.
No Nulls.
No inconsistencies.
Nothing raising concern.

### Table 11 - transactions

In [55]:
# View data size
transactions.shape

(50000, 8)

This is the meat of our data with 50,000 rows and 8 columns.

In [56]:
# View some of the data
transactions.head(10)

,TransactionID,AccountOriginID,AccountDestinationID,TransactionTypeID,Amount,TransactionDate,BranchID,Description
0,3022681,201164,200868,2,855.17,2023-04-20 02:00:00.000000,41,Transaction 22681
1,3037846,200138,201402,2,806.20,2021-08-10 15:00:00.000000,43,Transaction 37846
2,3045293,201002,201180,1,1229.44,2020-08-16 03:00:00.000000,5,Transaction 45293
3,3017397,201066,201144,4,4441.60,2021-10-10 06:00:00.000000,14,Transaction 17397
4,3016750,200289,201413,3,2526.20,2022-07-28 00:00:00.000000,37,Transaction 16750
5,3020141,200144,200319,3,1498.48,2023-07-08 13:00:00.000000,20,Transaction 20141
6,3039836,201137,200095,1,2556.13,2021-09-03 01:00:00.000000,34,Transaction 39836
7,3018170,201594,200292,2,2321.85,2021-07-18 21:00:00.000000,14,Transaction 18170
8,3048888,201340,200210,3,1649.81,2022-06-04 01:00:00.000000,50,Transaction 48888
9,3048575,200503,201417,1,632.38,2021-02-22 19:00:00.000000,22,Transaction 48575


In [57]:
# Review data types and nulls
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   TransactionID         50000 non-null  int64  
 1   AccountOriginID       50000 non-null  int64  
 2   AccountDestinationID  50000 non-null  int64  
 3   TransactionTypeID     50000 non-null  int64  
 4   Amount                50000 non-null  float64
 5   TransactionDate       49000 non-null  object 
 6   BranchID              50000 non-null  int64  
 7   Description           50000 non-null  object 
dtypes: float64(1), int64(5), object(2)
memory usage: 3.1+ MB


Nulls noted.
TransactionDate needs to be reformatted to be consistent with other data fields otherwise datatypes appear appropriate.

In [58]:
# Review for nulls
transactions.isnull().sum()

TransactionID              0
AccountOriginID            0
AccountDestinationID       0
TransactionTypeID          0
Amount                     0
TransactionDate         1000
BranchID                   0
Description                0
dtype: int64

1000/50000 is 2% of the overall population. Some may be cleaned up during the duplicate clean up. 

Depending on what the TransactionDate represents it could be valid to have a NaN. If it is based on settlement, these transactions may not have settled yet and are pending. 

This would be a question to ask the business.

In [ ]:
# Review data stats
transactions.describe()

AccountOriginID and AccountDestinationID appears consistent with accounts table.

BranchID appears close to branches table. 

TransactionTypeID appears consistent with transaction table.

Amount being positive appears appropriate based on how to and from accounts are indicated.

Nothing is raising any flags at this time in the data.

In [60]:
# Sort data to see if dates appear appropriate
## By TransactionID
transactions.sort_values(by=['TransactionID']).head(10)

,TransactionID,AccountOriginID,AccountDestinationID,TransactionTypeID,Amount,TransactionDate,BranchID,Description
22482,3000000,201241,200847,3,2984.02,2023-12-29 10:00:00.000000,26,Transaction 0
7611,3000001,201103,200262,3,4713.48,2023-05-12 02:00:00.000000,23,Transaction 1
6872,3000002,201563,200589,3,1600.23,2020-07-14 17:00:00.000000,5,Transaction 2
19463,3000003,200512,200404,1,4279.61,2023-09-09 07:00:00.000000,36,Transaction 3
45922,3000004,201107,201408,2,4125.32,2022-05-29 02:00:00.000000,44,Transaction 4
18410,3000005,200353,201615,3,2445.17,2023-07-14 03:00:00.000000,26,Transaction 5
45960,3000006,200822,200492,3,3592.37,2020-11-23 23:00:00.000000,14,Transaction 6
42915,3000007,200255,200152,1,3291.83,2020-04-16 21:00:00.000000,40,Transaction 7
25253,3000008,200379,200993,1,2965.03,2022-10-10 14:00:00.000000,47,Transaction 8
10979,3000009,201103,200376,3,611.83,2023-01-10 11:00:00.000000,47,Transaction 9


The TransactionsIDs are not showing sequential order based on Dates. This could be they are settlement dates and not origination dates.

Seems odd that a transaction would take years to settle if TransactionDate is based on settlement. Especially if this is within the same organization.

Is it possible items are being backdated?

In [ ]:
# Review for duplicates
transactions.duplicated().sum()

np.int64(500)

Duplicated transactions noted. 500 < 1000 thus only some of the NaN TransactionDate would possibly be cleaned with this.

In [151]:
# View select duplicates
transactions[transactions.duplicated(keep=False)].sort_values(by=['TransactionID']).head(10)

,TransactionID,AccountOriginID,AccountDestinationID,TransactionTypeID,Amount,TransactionDate,BranchID,Description
33986,3000044,201292,200088,2,1644.10,2022-11-21 11:00:00.000000,26,Transaction 44
27816,3000044,201292,200088,2,1644.10,2022-11-21 11:00:00.000000,26,Transaction 44
42868,3000051,200946,201528,1,1127.32,2024-01-27 18:00:00.000000,32,Transaction 51
27578,3000051,200946,201528,1,1127.32,2024-01-27 18:00:00.000000,32,Transaction 51
34758,3000098,200673,201509,2,175.29,2021-01-02 00:00:00.000000,48,Transaction 98
44486,3000098,200673,201509,2,175.29,2021-01-02 00:00:00.000000,48,Transaction 98
6384,3000145,200232,201371,4,4700.18,2025-10-04 15:01:43.084879,25,Transaction 145
34004,3000145,200232,201371,4,4700.18,2025-10-04 15:01:43.084879,25,Transaction 145
19838,3000164,200756,201620,1,2281.73,2023-08-04 07:00:00.000000,42,Transaction 164
43793,3000164,200756,201620,1,2281.73,2023-08-04 07:00:00.000000,42,Transaction 164


If we removed duplicated transactions, will the account balances reconcile? Is the dataset even built to allow for this? No notes in readme file with the data download.

If able, this would be something to explore more to ensure data accuracy.

In [ ]:
# View the invalid dates
pd.to_datetime(transactions['TransactionDate'], errors='coerce').dt.date.isna().value_counts()

TransactionDate
False    49000
True      1000
Name: count, dtype: int64

1000 is consistent with the nulls noted previously. Does not appear clean up is needed.

In [ ]:
# Sequence Check
## Create expected range
TransactionID_range = set(range(transactions['TransactionID'].min(),transactions['TransactionID'].max() + 1))

## Compare actual to expected 
missing_TransactionIDs = TransactionID_range - set(transactions['TransactionID'])

## View missing sequences
print(f"Missing TransactionIDs: {list(missing_TransactionIDs)}")

Missing TransactionIDs: []


No missing Ids.

# Transformation
The final production cleaning logic is maintained in transform.py. 

The notebook retains this information for documentation purposes.

## Data Cleaning
Based on information obtained through the initial data exploration process, in this section, we will do the clean up of the data.

### Table 1 - account_statuses
No cleaning was deemed necessary.

### Table 2 - account_types
No cleaning was deemed necessary.

### Table 3 - accounts
During data exploration we found:
-   OpeningDate needs to be reformatted to date without time.
-   Duplicates need to be removed.

In [71]:
# Remove duplicates and assign to new dataset to preserve original
accounts_cleaned = accounts.drop_duplicates().copy()

In [72]:
# Format date
accounts_cleaned['OpeningDate'] = pd.to_datetime(accounts_cleaned['OpeningDate'], errors='coerce').dt.date

In [73]:
# Check data for duplicates
print(accounts_cleaned.duplicated().value_counts())

## None noted which is as expected. Records are expected to be 1651.

False    1651
Name: count, dtype: int64


In [152]:
# Check date formatting
accounts_cleaned.head()

,AccountID,CustomerID,AccountTypeID,AccountStatusID,Balance,OpeningDate
0,200094,10123,3,1,48348.54,2018-06-12
1,201108,10077,3,1,35001.41,2019-10-30
2,201453,10321,3,2,57081.03,2020-05-24
3,200581,10871,5,1,63164.33,2021-01-27
4,200003,10765,1,1,58739.64,2018-09-12


In [91]:
# Check the raw data to make sure unchanged
accounts.head()

,AccountID,CustomerID,AccountTypeID,AccountStatusID,Balance,OpeningDate
0,200094,10123,3,1,48348.54,2018-06-12 00:00:00.000000
1,201108,10077,3,1,35001.41,2019-10-30 00:00:00.000000
2,201453,10321,3,2,57081.03,2020-05-24 00:00:00.000000
3,200581,10871,5,1,63164.33,2021-01-27 00:00:00.000000
4,200003,10765,1,1,58739.64,2018-09-12 00:00:00.000000


### Table 4 - addresses

During data exploration the following was identified for cleaning:
- Duplicate values
- Inconsistent values in the Country field

Also, noted:
- Missing squence addressID 200000 to be explored further. Why is this missing and does it exist in any other tables? This could create an issue when creating keys in SQL.

In [79]:
# Create new dataset and remove duplicates
addresses_cleaned = addresses.drop_duplicates().copy()

In [80]:
# Check data for duplicates
addresses_cleaned.duplicated().value_counts()

False    1210
Name: count, dtype: int64

No duplicates noted

In [81]:
# Update values in Country field to be consistent
addresses_cleaned.loc[~addresses_cleaned.Country.isin(['United States']), 'Country'] = 'United States' 


In [ ]:
# Check Country
addresses_cleaned['Country'].value_counts()


Country
United States    1210
Name: count, dtype: int64

Values are now consistent

### Table 5 - branches
No cleaning deemed necessary.

### Table 6 - customer_types
No cleaning deemed necessary.

### Table 7 - customers
During data exploration the following was identified for cleaning:
- Duplicate values
- Invalid DateofBirth values and formatting needs updating

In [140]:
# Create new dataset and remove duplicates to not override original data
customers_cleaned = customers.drop_duplicates().copy()

In [141]:
# Check data for duplicates
customers_cleaned.duplicated().value_counts()

False    1100
Name: count, dtype: int64

No duplicates noted as expected.

In [142]:
# Update invalid dates using replace

customers_cleaned['DateOfBirth'] = (
    customers_cleaned['DateOfBirth']
    .replace({
        '1960-08-27T00:00:00' : '1960-08-27 00:00:00.000000',
        'NaT' : np.nan,
        '07.04.1981' : '1981-07-04 00:00:00.000000',
        '1985/07/09' : '1985-07-09 00:00:00.000000',
        '02/11/1970' : '1970-02-11 00:00:00.000000',
        '09/12/1982' : '1982-09-12 00:00:00.000000',
        '1989/03/17' : '1989-03-17 00:00:00.000000',
        '1962-15-01' : '1962-01-15 00:00:00.000000',
        '1974-31-01' : '1974-01-31 00:00:00.000000',
        '01.18.1960' : '1960-01-18 00:00:00.000000',
        '1969/08/28' : '1969-08-28 00:00:00.000000'
    })
)


In [143]:
# View the invalid dates to check data before continuing
customers_cleaned[pd.to_datetime(customers_cleaned['DateOfBirth'], errors='coerce').dt.date.isna()]

,CustomerID,FirstName,LastName,DateOfBirth,AddressID,CustomerTypeID
124,10626,Johnson,Lawrence,NaN,143,2
199,10226,Thurman,NaN,NaN,1020,2
205,10713,Gilbert,Bernard,NaN,1139,1
278,10653,Faviola,Robertson,NaN,1067,2
279,10505,Audrie,Juarez,NaN,779,1
421,10515,Eloy,Chaney,NaN,373,3
470,11002,Bryan,NaN,NaN,534,3
477,10189,Ula,Bupns,NaN,603,3
481,10377,Magaly,Mccullough,NaN,333,2
487,10690,Emerita,Villarreal,NaN,59,3


Only seeing Null values as expected.

In [144]:
# Format date
## Note this had to be done after other values are replaced or it will replace all invalid items with NaN
customers_cleaned['DateOfBirth'] = pd.to_datetime(customers_cleaned['DateOfBirth'], errors='coerce').dt.date

In [145]:
# Check Data
customers_cleaned.head()

,CustomerID,FirstName,LastName,DateOfBirth,AddressID,CustomerTypeID
0,10832,Nyla,Aguirre,1974-02-07,881,1
1,10983,NaN,Battle,1963-02-01,958,2
2,10837,Angelena,Harrington,1964-03-25,86,3
3,10107,Remona,Glass,1965-09-16,595,1
4,10553,King,Becker,1966-02-20,969,3


Dates look formatted correctly.

In [146]:
# Check Nulls
customers_cleaned['DateOfBirth'].isnull().sum()


np.int64(22)

32 invalid prior, replaced 10, leaves 22 'NaT' replaced as Null (NaN). Null values for DateOfBirth is as expected.

### Table 8 - loan_statuses
No cleaning deemed necessary.

### Table 9 - loans
During data exploration the following was identified for cleaning:
- Duplicate values
- StartDate and EstimatedEndDate needs reformatting
- One invalid EstimatedEndDate (Start was after End) needs to be updated to imputed average 

In [ ]:
# Create new dataset and remove duplicates to not override original data
loans_cleaned = loans.drop_duplicates().copy()

In [ ]:
# Check for duplicates
loans_cleaned.duplicated().value_counts()

False    330
Name: count, dtype: int64

Duplicates removed appropriately.

In [45]:
# Format date fields using function

for col in ['StartDate', 'EstimatedEndDate']:
    loans_cleaned[col] = pd.to_datetime(loans_cleaned[col], errors='coerce').dt.date

In [92]:
# Check data
loans_cleaned.head()

,LoanID,AccountID,LoanStatusID,PrincipalAmount,InterestRate,StartDate,EstimatedEndDate
0,400230,200876,1,76958.56,0.0547,2022-11-20,2026-08-06
1,400307,200789,1,29013.67,0.0321,2022-02-22,2025-12-08
2,400233,201275,1,48596.76,0.1017,2021-11-21,2023-07-30
3,400100,200070,1,9191.43,0.0999,2021-08-14,2023-09-18
4,400141,200808,1,76322.83,0.0906,2021-06-04,2024-10-23


In [93]:
# Impute and update value for invalid EstimatedEndDate

## Create a dataset with only valid loan periods 
Valid_Periods = loans_cleaned[loans_cleaned['StartDate'] <= loans_cleaned['EstimatedEndDate']]

Valid_Periods.head()

,LoanID,AccountID,LoanStatusID,PrincipalAmount,InterestRate,StartDate,EstimatedEndDate
0,400230,200876,1,76958.56,0.0547,2022-11-20,2026-08-06
1,400307,200789,1,29013.67,0.0321,2022-02-22,2025-12-08
2,400233,201275,1,48596.76,0.1017,2021-11-21,2023-07-30
3,400100,200070,1,9191.43,0.0999,2021-08-14,2023-09-18
4,400141,200808,1,76322.83,0.0906,2021-06-04,2024-10-23


In [94]:
## Calculate individual row differences and then get the overall average
Loan_Days_Mean = (pd.to_datetime(Valid_Periods['EstimatedEndDate']) - pd.to_datetime(Valid_Periods['StartDate'])).dt.days.mean()

Loan_Days_Mean


np.float64(1125.6446540880504)

In [49]:
## Update invalid dates

loans_cleaned['EstimatedEndDate'] = np.where(
    pd.to_datetime(loans_cleaned['StartDate']) > pd.to_datetime(loans_cleaned['EstimatedEndDate']),
    pd.to_datetime(loans_cleaned['StartDate']) + pd.to_timedelta(Loan_Days_Mean, unit='D'),
    pd.to_datetime(loans_cleaned['EstimatedEndDate'])
)


In [153]:
# Check data
loans_cleaned.head()

,LoanID,AccountID,LoanStatusID,PrincipalAmount,InterestRate,StartDate,EstimatedEndDate
0,400230,200876,1,76958.56,0.0547,2022-11-20,2026-08-06
1,400307,200789,1,29013.67,0.0321,2022-02-22,2025-12-08
2,400233,201275,1,48596.76,0.1017,2021-11-21,2023-07-30
3,400100,200070,1,9191.43,0.0999,2021-08-14,2023-09-18
4,400141,200808,1,76322.83,0.0906,2021-06-04,2024-10-23


In [ ]:
# Check for invalid dates
loans_cleaned[loans_cleaned['StartDate'] > loans_cleaned['EstimatedEndDate']]

,LoanID,AccountID,LoanStatusID,PrincipalAmount,InterestRate,StartDate,EstimatedEndDate


None noted as expected

### Table 10 - transaction_types

No cleaning deemed necessary.

### Table 11 - transactions
During data exploration we found:
-   TransactionDate needs to be reformatted to date without time.
-   Duplicates need to be removed.

In [95]:
# Create new dataset and remove duplicates to not override original data
transactions_cleaned = transactions.drop_duplicates().copy()

In [96]:
# Check for duplicates
transactions_cleaned.duplicated().value_counts()

False    49500
Name: count, dtype: int64

Duplicates removed appropriately. Noted 50,000 records orignally less 500 duplicates = 49,500

In [97]:
# Format date
transactions_cleaned['TransactionDate'] = pd.to_datetime(transactions_cleaned['TransactionDate'], errors='coerce').dt.date

In [98]:
# Check data
transactions_cleaned.head()

,TransactionID,AccountOriginID,AccountDestinationID,TransactionTypeID,Amount,TransactionDate,BranchID,Description
0,3022681,201164,200868,2,855.17,2023-04-20,41,Transaction 22681
1,3037846,200138,201402,2,806.20,2021-08-10,43,Transaction 37846
2,3045293,201002,201180,1,1229.44,2020-08-16,5,Transaction 45293
3,3017397,201066,201144,4,4441.60,2021-10-10,14,Transaction 17397
4,3016750,200289,201413,3,2526.20,2022-07-28,37,Transaction 16750


# Export
See export_cleaned_data.py

In [148]:
# Export datasets
## account_statuses.to_csv('account_statuses_cleaned.csv', index=False)
## account_types.to_csv('account_types_cleaned.csv', index=False)
## accounts_cleaned.to_csv("accounts_cleaned.csv", index=False)
## addresses_cleaned.to_csv('addresses_cleaned.csv', index=False)
## branches.to_csv('branches_cleaned.csv', index=False)
## customer_types.to_csv('customer_types_cleaned.csv', index=False)
## customers_cleaned.to_csv('customers_cleaned.csv', index=False)
## loan_statuses.to_csv('loan_statuses_cleaned.csv', index=False)
## loans_cleaned.to_csv('loans_cleaned.csv', index=False)
## transaction_types.to_csv('transaction_types_cleaned.csv', index=False)
## transactions_cleaned.to_csv('transactions_cleaned.csv', index=False) 

# LOAD
See Financial Transaction ETL LOAD.sql.

In [149]:
# Run shapes for validation purposes during load
print('account_statuses:',account_statuses.shape)
print('account_types:',account_types.shape)
print('accounts:',accounts_cleaned.shape)
print('addresses:',addresses_cleaned.shape)
print('branches:',branches.shape)
print('customer_types:',customer_types.shape)
print('customers:',customers_cleaned.shape)
print('loan_statuses:',account_statuses.shape)
print('loans:',loans_cleaned.shape)
print('transaction_types:',transaction_types.shape)
print('transactions:',transactions_cleaned.shape)

account_statuses: (3, 2)
account_types: (5, 2)
accounts: (1651, 6)
addresses: (1210, 4)
branches: (50, 3)
customer_types: (3, 2)
customers: (1100, 6)
loan_statuses: (3, 2)
loans: (330, 7)
transaction_types: (4, 2)
transactions: (49500, 8)
